<a href="https://colab.research.google.com/github/lmknijn/jeweled_style/blob/main/Polyptoton_adj_window_sizes_no_repetitions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# install language models
!pip install https://huggingface.co/chcaa/grc_odycy_joint_trf/resolve/main/grc_odycy_joint_trf-0.7.0-py3-none-any.whl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 497.3/497.3 MB 3.6 MB/s eta 0:00:00


In [ ]:
import spacy

In [ ]:
nlp = spacy.load("grc_odycy_joint_trf")

In [ ]:
# python packages
!pip install -q GitPython MyCapytain
!pip install -q git+https://github.com/cwf2/dices-client

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.9/70.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 13.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.2/697.2 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.7/193.7 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# utils
import os
import re
import json
import git
import requests
import unicodedata as ud

# DICES packages
from dicesapi import DicesAPI, SpeechGroup
from dicesapi.text import CtsAPI
import dicesapi.text

# for working with local CTS repositories
from MyCapytain.resolvers.cts.local import CtsCapitainsLocalResolver
from MyCapytain.resources.prototypes.metadata import UnknownCollection

# for analysis
import pandas as pd
import numpy as np

# for output
from IPython.display import HTML
import seaborn as sns
from matplotlib import pyplot as plt

In [ ]:
repo_names = ['canonical-greekLit']

print('Checking for local text repositories...')

for repo in repo_names:
    local_dir = os.path.join('data', repo)
    remote_url = f'https://github.com/cwf2/{repo}.git'

    if os.path.exists(local_dir):
        print(f' - {local_dir} exists!')
    else:
        print(f' - retrieving {remote_url}')
        git.Repo.clone_from(remote_url, local_dir)

Checking for local text repositories...
 - retrieving https://github.com/cwf2/canonical-greekLit.git


In [ ]:
api = DicesAPI(
    logfile = 'dices.log',
    logdetail = 0,
)

DEBUG:dicesLog:New log created with No Detail
DEBUG:dicesLog:Database Initialized


In [ ]:
# path to local repos
repo_paths = [os.path.join('data', repo) for repo in repo_names]

# create a local resolver
local_resolver = CtsCapitainsLocalResolver(repo_paths, logger=api.log)

# initialize the CTS API
cts = CtsAPI(dices_api = api)

# overwrite the default resolver
cts._resolvers = {None: local_resolver}

INFO:dicesLog:data/canonical-greekLit/data/tlg0001/tlg001/tlg0001.tlg001.perseus-grc2.xml has been parsed 
INFO:dicesLog:data/canonical-greekLit/data/tlg0641/tlg001/tlg0641.tlg001.perseus-grc2.xml has been parsed 
INFO:dicesLog:data/canonical-greekLit/data/tlg0008/tlg001/tlg0008.tlg001.perseus-grc4.xml has been parsed 
INFO:dicesLog:data/canonical-greekLit/data/tlg0008/tlg001/tlg0008.tlg001.perseus-grc3.xml has been parsed 
INFO:dicesLog:data/canonical-greekLit/data/tlg0008/tlg001/tlg0008.tlg001.perseus-eng2.xml has been parsed 
INFO:dicesLog:data/canonical-greekLit/data/tlg0033/tlg003/tlg0033.tlg003.perseus-grc2.xml has been parsed 
INFO:dicesLog:data/canonical-greekLit/data/tlg0033/tlg003/tlg0033.tlg003.perseus-eng2.xml has been parsed 
INFO:dicesLog:data/canonical-greekLit/data/tlg0033/tlg002/tlg0033.tlg002.perseus-grc2.xml has been parsed 
INFO:dicesLog:data/canonical-greekLit/data/tlg0033/tlg002/tlg0033.tlg002.perseus-eng3.xml has been parsed 
INFO:dicesLog:data/canonical-greekLit

In [ ]:
# URN for the text
urn = 'urn:cts:greekLit:tlg0647.tlg001'

# get a resolver
resolver = cts.getResolver(urn)

# request the whole text
passage = resolver.getTextualNode(urn)

In [ ]:
def create_id(book, line):
    '''create a string to represent locus
        - zero pad book number to 2 digits
        - zero pad line number to 3 digits
        - keep letter suffix
        - use underscore to separate book and line
    '''

    # check line for alpha suffix
    m = re.match(r"([0-9]+)([a-z]?)", line)         # ? means the letter is optional
    if m:
        numeric = m.group(1)
        letter = m.group(2)
    else:
        numeric = line
        letter = ""

    id = f"{int(book):02d}_{int(numeric):03d}{letter}"

    return id

For Il., Argon., PH and D.

In [ ]:
# xml representation of the text
xml = passage.xml

# empty list to hold the lines
lines = list()

# iterate over books
for book in xml.findall('.//div[@subtype="book"]', namespaces=xml.nsmap):  # capital B for Iliad
    bn = book.get('n')

    # remove notes
    for note in book.findall('.//l//note', namespaces=xml.nsmap):
        note.clear(keep_tail=True)

    # extract lines
    for line in book.findall('.//l', namespaces=xml.nsmap):

        # line number
        ln = line.get('n')

        # text
        text = ''.join(line.itertext())

        # clean up text
        text = re.sub(r'\s+', ' ', text).strip()

        # create unified id from book and line number
        id = create_id(bn, ln)


        # add to array
        lines.append(dict(
            id = id,
            book = bn,
            line = ln,
            text = text,
        ))

For Triphiodorus

In [ ]:
# xml representation of the text
xml = passage.xml

# empty list to hold the lines
lines = list()


bn = 0

# remove notes
for note in xml.findall('.//l//note', namespaces=xml.nsmap):
    note.clear(keep_tail=True)

# extract lines
for line in xml.findall('.//l', namespaces=xml.nsmap):

    # line number
    ln = line.get('n')

    # text
    text = ''.join(line.itertext())

    # clean up text
    text = re.sub(r'\s+', ' ', text).strip()

    # create unified id from book and line number
    id = create_id(bn, ln)


    # add to array
    lines.append(dict(
        id = id,
        book = bn,
        line = ln,
        text = text,
    ))

In [ ]:
lines = pd.DataFrame(lines)
display(lines)

,id,book,line,text
0,00_001,0,1,τέρμα πολυκμήτοιο μεταχρόνιον πολέμοιο
1,00_002,0,2,"καὶ λόχον, Ἀργείης ἱππήλατον ἔργον Ἀθήνης,"
2,00_003,0,3,αὐτίκα μοι σπεύδοντι πολὺν διὰ μῦθον ἀνεῖσα
3,00_004,0,4,"ἔννεπε, Καλλιόπεια, καὶ ἀρχαίην ἔριν ἀνδρῶν"
4,00_005,0,5,κεκριμένου πολέμοιο ταχείῃ λῦσον ἀοιδῇ.
...,...,...,...,...
686,00_687,0,687,υῆνιν ἱλασσάμενοι τεθνειότος Αἰακίδαο
687,00_688,0,688,"Τρωιάδας τε γυναῖκας ἐλάγχανον, ἄλλα τε πάντα"
688,00_689,0,689,χρυσὸν ἐμοιρήσαντο καὶ ἄργυρον· οἷσι βαθείας
689,00_690,0,690,νῆας ἐπαχθήσαντες ἐριγδούπου διὰ πόντου


In [ ]:
lines['tokens'] = lines['text'].apply(nlp)
display(lines)

,id,book,line,text,tokens
0,00_001,0,1,τέρμα πολυκμήτοιο μεταχρόνιον πολέμοιο,"(τέρμα, πολυκμήτοιο, μεταχρόνιον, πολέμοιο)"
1,00_002,0,2,"καὶ λόχον, Ἀργείης ἱππήλατον ἔργον Ἀθήνης,","(καὶ, λόχον, ,, Ἀργείης, ἱππήλατον, ἔργον, Ἀθή..."
2,00_003,0,3,αὐτίκα μοι σπεύδοντι πολὺν διὰ μῦθον ἀνεῖσα,"(αὐτίκα, μοι, σπεύδοντι, πολὺν, διὰ, μῦθον, ἀν..."
3,00_004,0,4,"ἔννεπε, Καλλιόπεια, καὶ ἀρχαίην ἔριν ἀνδρῶν","(ἔννεπε, ,, Καλλιόπεια, ,, καὶ, ἀρχαίην, ἔριν,..."
4,00_005,0,5,κεκριμένου πολέμοιο ταχείῃ λῦσον ἀοιδῇ.,"(κεκριμένου, πολέμοιο, ταχείῃ, λῦσον, ἀοιδῇ, .)"
...,...,...,...,...,...
686,00_687,0,687,υῆνιν ἱλασσάμενοι τεθνειότος Αἰακίδαο,"(υῆνιν, ἱλασσάμενοι, τεθνειότος, Αἰακίδαο)"
687,00_688,0,688,"Τρωιάδας τε γυναῖκας ἐλάγχανον, ἄλλα τε πάντα","(Τρωιάδας, τε, γυναῖκας, ἐλάγχανον, ,, ἄλλα, τ..."
688,00_689,0,689,χρυσὸν ἐμοιρήσαντο καὶ ἄργυρον· οἷσι βαθείας,"(χρυσὸν, ἐμοιρήσαντο, καὶ, ἄργυρον, ·, οἷσι, β..."
689,00_690,0,690,νῆας ἐπαχθήσαντες ἐριγδούπου διὰ πόντου,"(νῆας, ἐπαχθήσαντες, ἐριγδούπου, διὰ, πόντου)"


What needs to happen?

- Divide the tokens into lists of five
- Each token is the first of a new set of five (so 1-5, 2-6, 3-7 etc.)
- Keep each token's id, book and line number (maybe change id into 00_001a and so on?)


In [ ]:
window_size = 7

all_tokens_data = []
for index, row in lines.iterrows():
    for token in row['tokens']:
        all_tokens_data.append({
            'token_obj': token,
            'original_line_id': row['id'],
            'original_line_book': row['book'],
            'original_line_line': row['line']
        })

# Now create the 5-token sequences
n_token_sequences = []
for i in range(len(all_tokens_data) - (window_size - 1)): # Ensure there are at least 5 tokens to form a sequence
    # Get the 5 token objects
    sequence_tokens = [all_tokens_data[j]['token_obj'] for j in range(i, i + window_size)]

    # Get the metadata (id, book, line) from the first token in the sequence
    first_token_metadata = all_tokens_data[i]

    n_token_sequences.append({
        'id': first_token_metadata['original_line_id'],
        'book': first_token_metadata['original_line_book'],
        'line': first_token_metadata['original_line_line'],
        'n_tokens': sequence_tokens
    })

# Convert to DataFrame for easier handling and display
n_token_df = pd.DataFrame(n_token_sequences)

# Add an alphabetical suffix to the 'id' column
import string

new_ids = []
current_id = None
suffix_counter = 0

for index, row in n_token_df.iterrows():
    if row['id'] != current_id:
        current_id = row['id']
        suffix_counter = 0

    suffix_letter = string.ascii_lowercase[suffix_counter]
    new_id = f"{row['id']}{suffix_letter}"
    new_ids.append(new_id)

    suffix_counter += 1

n_token_df['id'] = new_ids

print(f"Created {len(n_token_df)} six-token sequences with unique IDs.")
display(n_token_df.head())

Created 4801 six-token sequences with unique IDs.


,id,book,line,n_tokens
0,00_001a,0,1,"[τέρμα, πολυκμήτοιο, μεταχρόνιον, πολέμοιο, κα..."
1,00_001b,0,1,"[πολυκμήτοιο, μεταχρόνιον, πολέμοιο, καὶ, λόχο..."
2,00_001c,0,1,"[μεταχρόνιον, πολέμοιο, καὶ, λόχον, ,, Ἀργείης..."
3,00_001d,0,1,"[πολέμοιο, καὶ, λόχον, ,, Ἀργείης, ἱππήλατον, ..."
4,00_002a,0,2,"[καὶ, λόχον, ,, Ἀργείης, ἱππήλατον, ἔργον, Ἀθή..."


Next cell: change so that each row is one token from a set of five, e.g. 00_001a

In [ ]:
# start with an empty list of rows
rows = []

# iterate over speeches
for line in n_token_df.itertuples():   # lines

    # iterate over tokens
    for token in line.n_tokens:   # tokens

        # create a record for this token
        row = {
            "window_id": line.id,
            "book": line.book,
            "line": line.line,
            "token": token.text,        #token.text
            "lemma": token.lemma_,
            "pos": token.pos_,
            "lemma_pos": (token.lemma_, token.pos_)
        }

        # add row to the list of rows
        rows.append(row)

# convert to a table
n_tokens = pd.DataFrame(rows)   # tokens

display(n_tokens[0:40])  # tokens

,window_id,book,line,token,lemma,pos,lemma_pos
0,00_001a,0,1,τέρμα,τέρμα,NOUN,"(τέρμα, NOUN)"
1,00_001a,0,1,πολυκμήτοιο,πολυκμήτοιο,ADJ,"(πολυκμήτοιο, ADJ)"
2,00_001a,0,1,μεταχρόνιον,μεταχρόνιος,ADJ,"(μεταχρόνιος, ADJ)"
3,00_001a,0,1,πολέμοιο,πόλεμος,NOUN,"(πόλεμος, NOUN)"
4,00_001a,0,1,καὶ,καί,CCONJ,"(καί, CCONJ)"
5,00_001a,0,1,λόχον,λόχος,NOUN,"(λόχος, NOUN)"
6,00_001a,0,1,",",",",PUNCT,"(,, PUNCT)"
7,00_001b,0,1,πολυκμήτοιο,πολυκμήτοιο,ADJ,"(πολυκμήτοιο, ADJ)"
8,00_001b,0,1,μεταχρόνιον,μεταχρόνιος,ADJ,"(μεταχρόνιος, ADJ)"
9,00_001b,0,1,πολέμοιο,πόλεμος,NOUN,"(πόλεμος, NOUN)"


In [ ]:
from collections import Counter

In [ ]:
results = []

pos_mask = n_tokens['pos'].isin(['ADJ', 'NOUN', 'NUM', 'PROPN', 'VERB'])

excluded_lemmas = ["ἀλλ'", "'ἀλλʼ", "ἀλλά", "ἀλλὰ", "ἀτάρ", "ἀτὰρ", "αὐτάρ", "αὐτὰρ", "ἤ", "ἢ", "ἠέ", "ἠὲ", "ἦε", "ἦέ", "ἠδ'", "ἠδʼ" "ἠδέ", "ἠδὲ", "ἤτ'", "ἤτʼ", "ἤτε", "ἤτοι", "μηδ'", "μηδʼ", "μηδέ", "μηδὲ", "μήθ'", "μήτ'", "μήτʼ", "μήτε", "μήτέ", "οὐδ'", "οὐδʼ", "οὐδέ", "οὐδὲ", "οὔθ'", "οὔθʼ", "οὔτ'", "οὔτʼ", "οὔτε", "οὔτέ", "ἀρ'", "ἀρʼ", "ἄρ'", "ἄρʼ", "ἂρ'", "ἂρʼ", "ἄρα", "ἄρά", "ῥ'", "ῥʼ", "ῥα", "ῥά", "αὖ", "γ'", "γʼ", "γε", "γέ", "γάρ", "γὰρ", "δαὶ", "δ'", "δʼ", "δέ", "δὲ", "θην", "θήν", "κε", "κέ", "κεν", "κέν", "χ'", "χʼ", "κ'", "κʼ", "μάν", "μὰν", "μέν", "μὲν", "μήν", "μὴν", "νυ", "νύ", "νυν", "περ", "πέρ", "μὰ", "καί", "καὶ", "μή", "μὴ", "οὐ", "οὐκ", "οὐχ", "οὔ", "οὖν", "ὦ", "ὤ", "ὢ", "που", "πού", "τὰρ", "τε", "τέ", "τ'", "τʼ", "θ'", "θʼ", "ἂν", "ἄν", "δή", "δὴ", "τοιγὰρ", "αἴ", "αἲ", "αἴθ'", "αἴθʼ", "αἴθε", "εἰ", "εἴ", "εἴθ'",  "εἴθʼ", "ἐπεί", "ἐπεὶ", "ἐπειδὰν", "ἐπειδὴ", "ἐπήν", "ἐπὴν", "εὖθ'", "εὖθʼ", "εὖτ'", "εὖτʼ", "εὖτε", "εὖτέ", "ἠύτ'", "ἠύτʼ", "ἠΰτ'", "ἠΰτʼ", "ἠΰτε", "ἦμος", "ἡνίκ", "ἵν'", "ἵνʼ", "ἵνα", "ἵνά", "ἤν", "ἢν", "ὅτ'", "ὅτʼ", "ὅτε", "ὡς", "ὥστ'", "ὥστʼ", "ὥστε", "ὅππως", "ὅπως", "εἷος", "εἷός", "ἕως", "ἧος", "ἧός", "ὄφρ'", "ὄφρʼ", "ὄφρα", "ὄφρά", "πρίν", "πρὶν", "ὁ", "ὅς"]

lexicon_mask = ~n_tokens['lemma'].isin(excluded_lemmas)

combined_mask = pos_mask & lexicon_mask

lemmas_by_window = n_tokens.loc[combined_mask].groupby('window_id').agg(
    book = ('book', 'first'),
    line = ('line', 'first'),
    lemmas = ('lemma', list),
    tokens = ('token', list),
    pos = ('pos', list),
    lemma_pos = ('lemma_pos', list)
    ).reset_index()

display(lemmas_by_window)


,window_id,book,line,lemmas,tokens,pos,lemma_pos
0,00_001a,0,1,"[τέρμα, πολυκμήτοιο, μεταχρόνιος, πόλεμος, λόχος]","[τέρμα, πολυκμήτοιο, μεταχρόνιον, πολέμοιο, λό...","[NOUN, ADJ, ADJ, NOUN, NOUN]","[(τέρμα, NOUN), (πολυκμήτοιο, ADJ), (μεταχρόνι..."
1,00_001b,0,1,"[πολυκμήτοιο, μεταχρόνιος, πόλεμος, λόχος, ἀργ...","[πολυκμήτοιο, μεταχρόνιον, πολέμοιο, λόχον, Ἀρ...","[ADJ, ADJ, NOUN, NOUN, ADJ]","[(πολυκμήτοιο, ADJ), (μεταχρόνιος, ADJ), (πόλε..."
2,00_001c,0,1,"[μεταχρόνιος, πόλεμος, λόχος, ἀργεῖος, ἱππήλατος]","[μεταχρόνιον, πολέμοιο, λόχον, Ἀργείης, ἱππήλα...","[ADJ, NOUN, NOUN, ADJ, ADJ]","[(μεταχρόνιος, ADJ), (πόλεμος, NOUN), (λόχος, ..."
3,00_001d,0,1,"[πόλεμος, λόχος, ἀργεῖος, ἱππήλατος, ἔργον]","[πολέμοιο, λόχον, Ἀργείης, ἱππήλατον, ἔργον]","[NOUN, NOUN, ADJ, ADJ, NOUN]","[(πόλεμος, NOUN), (λόχος, NOUN), (ἀργεῖος, ADJ..."
4,00_002a,0,2,"[λόχος, ἀργεῖος, ἱππήλατος, ἔργον, ἀθήνη]","[λόχον, Ἀργείης, ἱππήλατον, ἔργον, Ἀθήνης]","[NOUN, ADJ, ADJ, NOUN, NOUN]","[(λόχος, NOUN), (ἀργεῖος, ADJ), (ἱππήλατος, AD..."
...,...,...,...,...,...,...,...
4795,00_690b,0,690,"[ἐπαχθέω, ἐρίγδουπος, πόντος, τροία, ἀνάγω]","[ἐπαχθήσαντες, ἐριγδούπου, πόντου, Τροίης, ἀνά...","[VERB, PROPN, PROPN, NOUN, VERB]","[(ἐπαχθέω, VERB), (ἐρίγδουπος, PROPN), (πόντος..."
4796,00_690c,0,690,"[ἐρίγδουπος, πόντος, τροία, ἀνάγω, μόθος]","[ἐριγδούπου, πόντου, Τροίης, ἀνάγοντο, μόθον]","[PROPN, PROPN, NOUN, VERB, NOUN]","[(ἐρίγδουπος, PROPN), (πόντος, PROPN), (τροία,..."
4797,00_690d,0,690,"[πόντος, τροία, ἀνάγω, μόθος, τελέω]","[πόντου, Τροίης, ἀνάγοντο, μόθον, τελέσαντες]","[PROPN, NOUN, VERB, NOUN, VERB]","[(πόντος, PROPN), (τροία, NOUN), (ἀνάγω, VERB)..."
4798,00_690e,0,690,"[πόντος, τροία, ἀνάγω, μόθος, τελέω, ἀχαιός]","[πόντου, Τροίης, ἀνάγοντο, μόθον, τελέσαντες, ...","[PROPN, NOUN, VERB, NOUN, VERB, ADJ]","[(πόντος, PROPN), (τροία, NOUN), (ἀνάγω, VERB)..."


In [ ]:
results = []

results.append(
      lemmas_by_window.groupby('window_id').agg(
      book = ('book', 'first'),
      line = ('line', 'first'),
      tokens = ('tokens', sum),
      lemmas = ('lemmas', sum),
      pos = ('pos', sum),
      lemma_pos = ('lemma_pos', sum)
      ))

results = pd.concat(results).sort_values('window_id').reset_index()

results
results['wc'] = results['lemma_pos'].apply(Counter)
results

,window_id,book,line,tokens,lemmas,pos,lemma_pos,wc
0,00_001a,0,1,"[τέρμα, πολυκμήτοιο, μεταχρόνιον, πολέμοιο, λό...","[τέρμα, πολυκμήτοιο, μεταχρόνιος, πόλεμος, λόχος]","[NOUN, ADJ, ADJ, NOUN, NOUN]","[(τέρμα, NOUN), (πολυκμήτοιο, ADJ), (μεταχρόνι...","{('τέρμα', 'NOUN'): 1, ('πολυκμήτοιο', 'ADJ'):..."
1,00_001b,0,1,"[πολυκμήτοιο, μεταχρόνιον, πολέμοιο, λόχον, Ἀρ...","[πολυκμήτοιο, μεταχρόνιος, πόλεμος, λόχος, ἀργ...","[ADJ, ADJ, NOUN, NOUN, ADJ]","[(πολυκμήτοιο, ADJ), (μεταχρόνιος, ADJ), (πόλε...","{('πολυκμήτοιο', 'ADJ'): 1, ('μεταχρόνιος', 'A..."
2,00_001c,0,1,"[μεταχρόνιον, πολέμοιο, λόχον, Ἀργείης, ἱππήλα...","[μεταχρόνιος, πόλεμος, λόχος, ἀργεῖος, ἱππήλατος]","[ADJ, NOUN, NOUN, ADJ, ADJ]","[(μεταχρόνιος, ADJ), (πόλεμος, NOUN), (λόχος, ...","{('μεταχρόνιος', 'ADJ'): 1, ('πόλεμος', 'NOUN'..."
3,00_001d,0,1,"[πολέμοιο, λόχον, Ἀργείης, ἱππήλατον, ἔργον]","[πόλεμος, λόχος, ἀργεῖος, ἱππήλατος, ἔργον]","[NOUN, NOUN, ADJ, ADJ, NOUN]","[(πόλεμος, NOUN), (λόχος, NOUN), (ἀργεῖος, ADJ...","{('πόλεμος', 'NOUN'): 1, ('λόχος', 'NOUN'): 1,..."
4,00_002a,0,2,"[λόχον, Ἀργείης, ἱππήλατον, ἔργον, Ἀθήνης]","[λόχος, ἀργεῖος, ἱππήλατος, ἔργον, ἀθήνη]","[NOUN, ADJ, ADJ, NOUN, NOUN]","[(λόχος, NOUN), (ἀργεῖος, ADJ), (ἱππήλατος, AD...","{('λόχος', 'NOUN'): 1, ('ἀργεῖος', 'ADJ'): 1, ..."
...,...,...,...,...,...,...,...,...
4795,00_690b,0,690,"[ἐπαχθήσαντες, ἐριγδούπου, πόντου, Τροίης, ἀνά...","[ἐπαχθέω, ἐρίγδουπος, πόντος, τροία, ἀνάγω]","[VERB, PROPN, PROPN, NOUN, VERB]","[(ἐπαχθέω, VERB), (ἐρίγδουπος, PROPN), (πόντος...","{('ἐπαχθέω', 'VERB'): 1, ('ἐρίγδουπος', 'PROPN..."
4796,00_690c,0,690,"[ἐριγδούπου, πόντου, Τροίης, ἀνάγοντο, μόθον]","[ἐρίγδουπος, πόντος, τροία, ἀνάγω, μόθος]","[PROPN, PROPN, NOUN, VERB, NOUN]","[(ἐρίγδουπος, PROPN), (πόντος, PROPN), (τροία,...","{('ἐρίγδουπος', 'PROPN'): 1, ('πόντος', 'PROPN..."
4797,00_690d,0,690,"[πόντου, Τροίης, ἀνάγοντο, μόθον, τελέσαντες]","[πόντος, τροία, ἀνάγω, μόθος, τελέω]","[PROPN, NOUN, VERB, NOUN, VERB]","[(πόντος, PROPN), (τροία, NOUN), (ἀνάγω, VERB)...","{('πόντος', 'PROPN'): 1, ('τροία', 'NOUN'): 1,..."
4798,00_690e,0,690,"[πόντου, Τροίης, ἀνάγοντο, μόθον, τελέσαντες, ...","[πόντος, τροία, ἀνάγω, μόθος, τελέω, ἀχαιός]","[PROPN, NOUN, VERB, NOUN, VERB, ADJ]","[(πόντος, PROPN), (τροία, NOUN), (ἀνάγω, VERB)...","{('πόντος', 'PROPN'): 1, ('τροία', 'NOUN'): 1,..."


In [ ]:
results_limited = results.drop(columns = ['lemmas', 'pos', 'lemma_pos'])

results_limited


,window_id,book,line,tokens,wc
0,00_001a,0,1,"[τέρμα, πολυκμήτοιο, μεταχρόνιον, πολέμοιο, λό...","{('τέρμα', 'NOUN'): 1, ('πολυκμήτοιο', 'ADJ'):..."
1,00_001b,0,1,"[πολυκμήτοιο, μεταχρόνιον, πολέμοιο, λόχον, Ἀρ...","{('πολυκμήτοιο', 'ADJ'): 1, ('μεταχρόνιος', 'A..."
2,00_001c,0,1,"[μεταχρόνιον, πολέμοιο, λόχον, Ἀργείης, ἱππήλα...","{('μεταχρόνιος', 'ADJ'): 1, ('πόλεμος', 'NOUN'..."
3,00_001d,0,1,"[πολέμοιο, λόχον, Ἀργείης, ἱππήλατον, ἔργον]","{('πόλεμος', 'NOUN'): 1, ('λόχος', 'NOUN'): 1,..."
4,00_002a,0,2,"[λόχον, Ἀργείης, ἱππήλατον, ἔργον, Ἀθήνης]","{('λόχος', 'NOUN'): 1, ('ἀργεῖος', 'ADJ'): 1, ..."
...,...,...,...,...,...
4795,00_690b,0,690,"[ἐπαχθήσαντες, ἐριγδούπου, πόντου, Τροίης, ἀνά...","{('ἐπαχθέω', 'VERB'): 1, ('ἐρίγδουπος', 'PROPN..."
4796,00_690c,0,690,"[ἐριγδούπου, πόντου, Τροίης, ἀνάγοντο, μόθον]","{('ἐρίγδουπος', 'PROPN'): 1, ('πόντος', 'PROPN..."
4797,00_690d,0,690,"[πόντου, Τροίης, ἀνάγοντο, μόθον, τελέσαντες]","{('πόντος', 'PROPN'): 1, ('τροία', 'NOUN'): 1,..."
4798,00_690e,0,690,"[πόντου, Τροίης, ἀνάγοντο, μόθον, τελέσαντες, ...","{('πόντος', 'PROPN'): 1, ('τροία', 'NOUN'): 1,..."


In [ ]:
# Identify repeated lemmas (appear at least twice)
results_limited['repeated_lemma'] = results_limited['wc'].apply(                                  # apply a lambda function to the data in wc in results
    lambda c: [lemma for lemma, count in c.items() if count >= 2]                  # a lambda function is an anonymous function, c is the Count object from wc
                                                                                   # If the count in the items (the lemma: count pairs in wc) is two or higher, keep the lemma
)

# Keep couplets where there is at least one such lemma
filtered_results = results_limited[results_limited['repeated_lemma'].apply(len) >= 1].copy()            # from the results, keep only the lines that have one or more from the list of repeated_lemmas

# Create a new DataFrame with one row per repeated lemma
expanded_results = filtered_results.explode('repeated_lemma')

# Extract lemma and pos from the tuple in 'repeated_lemma'
expanded_results['rep_lemma'] = expanded_results['repeated_lemma'].apply(lambda x: x[0])
expanded_results['rep_pos'] = expanded_results['repeated_lemma'].apply(lambda x: x[1])

# Drop the original 'repeated_lemma' column and other intermediate columns
expanded_results = expanded_results.drop(columns=['repeated_lemma'])

# Display the results
display(expanded_results)

,window_id,book,line,tokens,wc,rep_lemma,rep_pos
3122,00_448c,0,448,"[ἐπίδημος, ἀμήχανος, ὕβρις, ὕβρις]","{('ἐπίδημος', 'ADJ'): 1, ('ἀμήχανος', 'ADJ'): ...",ὕβρις,NOUN
3123,00_448d,0,448,"[ἀμήχανος, ὕβρις, ὕβρις, ἐλαφρίζουσα]","{('ἀμήχανος', 'ADJ'): 1, ('ὕβρις', 'NOUN'): 2,...",ὕβρις,NOUN
3124,00_448e,0,448,"[ἀμήχανος, ὕβρις, ὕβρις, ἐλαφρίζουσα, μέθην]","{('ἀμήχανος', 'ADJ'): 1, ('ὕβρις', 'NOUN'): 2,...",ὕβρις,NOUN
3125,00_448f,0,448,"[ἀμήχανος, ὕβρις, ὕβρις, ἐλαφρίζουσα, μέθην, λ...","{('ἀμήχανος', 'ADJ'): 1, ('ὕβρις', 'NOUN'): 2,...",ὕβρις,NOUN
3126,00_448g,0,448,"[ὕβρις, ὕβρις, ἐλαφρίζουσα, μέθην, λυσήνορος, ...","{('ὕβρις', 'NOUN'): 2, ('ἐλαφρίζω', 'VERB'): 1...",ὕβρις,NOUN
3668,00_527g,0,527,"[νῆες, ὠκύτεραι, κραιπνῶν, ἀνέμων, ἀνέμων]","{('ναῦς', 'NOUN'): 1, ('ὠκύτατος', 'ADJ'): 1, ...",ἄνεμος,NOUN
3669,00_527h,0,527,"[νῆες, ὠκύτεραι, κραιπνῶν, ἀνέμων, ἀνέμων, ταχ...","{('ναῦς', 'NOUN'): 1, ('ὠκύτατος', 'ADJ'): 1, ...",ἄνεμος,NOUN
3670,00_527i,0,527,"[νῆες, ὠκύτεραι, κραιπνῶν, ἀνέμων, ἀνέμων, ταχ...","{('ναῦς', 'NOUN'): 1, ('ὠκύτατος', 'ADJ'): 1, ...",ἄνεμος,NOUN
3671,00_528a,0,528,"[ὠκύτεραι, κραιπνῶν, ἀνέμων, ἀνέμων, ταχυπειθέ...","{('ὠκύτατος', 'ADJ'): 1, ('κραιπνός', 'ADJ'): ...",ἄνεμος,NOUN
3672,00_528b,0,528,"[κραιπνῶν, ἀνέμων, ἀνέμων, ταχυπειθέι, ῥιπῇ, Ἴ...","{('κραιπνός', 'ADJ'): 1, ('ἄνεμος', 'NOUN'): 2...",ἄνεμος,NOUN


Exclude rows with repetitions (two identical tokens instead of two different forms of the same lemma)

In [ ]:
def exclude_repetitions(tokens_list):
    duplicates = set()
    for token in tokens_list:
        if token in duplicates:
            return True
        duplicates.add(token)
    return False

# Filter out rows where the 'tokens' list contains duplicate forms
noreps_expanded_results = expanded_results[~expanded_results['tokens'].apply(exclude_repetitions)].copy()

display(noreps_expanded_results)

,window_id,book,line,tokens,wc,rep_lemma,rep_pos
3844,00_554a,0,554,"[ἤθελεν, ἐχόλωσε, ἐθέλοντα]","{('ἐθέλω', 'VERB'): 2, ('χολόω', 'VERB'): 1}",ἐθέλω,VERB
4588,00_660c,0,660,"[πατρίδος, γαίης, γαῖα]","{('πατρίς', 'ADJ'): 1, ('γαῖα', 'NOUN'): 2}",γαῖα,NOUN
4589,00_660d,0,660,"[πατρίδος, γαίης, γαῖα, περιπτύξασα]","{('πατρίς', 'ADJ'): 1, ('γαῖα', 'NOUN'): 2, ('...",γαῖα,NOUN
4590,00_660e,0,660,"[πατρίδος, γαίης, γαῖα, περιπτύξασα, κεχηνότι]","{('πατρίς', 'ADJ'): 1, ('γαῖα', 'NOUN'): 2, ('...",γαῖα,NOUN
4591,00_660f,0,660,"[πατρίδος, γαίης, γαῖα, περιπτύξασα, κεχηνότι,...","{('πατρίς', 'ADJ'): 1, ('γαῖα', 'NOUN'): 2, ('...",γαῖα,NOUN
4592,00_660g,0,660,"[γαίης, γαῖα, περιπτύξασα, κεχηνότι, δέξατο, κ...","{('γαῖα', 'NOUN'): 2, ('περιπτύχω', 'VERB'): 1...",γαῖα,NOUN
4593,00_660h,0,660,"[γαίης, γαῖα, περιπτύξασα, κεχηνότι, δέξατο, κ...","{('γαῖα', 'NOUN'): 2, ('περιπτύχω', 'VERB'): 1...",γαῖα,NOUN


In [ ]:
noreps_expanded_results.to_csv('ph_pol_noreps_7win_12_3_26.csv')